# Notebook 2 Audio Feature Extraction

**Project:** AI Music Technical Quality Detector  
**Author:** Ankit Sharda · MSc AI & Data Science · University of Hull

---

## What this notebook does

This is the first analysis notebook in the pipeline. It reads every MP3 file from the collected dataset (human songs and AI-generated songs) and extracts measurable audio features for use in machine learning models.


Features are organised into six groups:

| Group | Features | What defect it can reveal |
|---|---|---|
| **Loudness** | LUFS, true peak, RMS, crest factor | Over-compression, clipping |
| **Dynamic range** | DR estimate, frame RMS spread | Flat/lifeless mastering |
| **Spectral** | Centroid, rolloff, flatness, flux, bandwidth, contrast, MFCCs 1-13, delta MFCCs, chroma | Brightness, muddiness, harshness, timbre |
| **Temporal** | BPM, beat regularity, onset strength, ZCR, silence ratio | Mechanical rhythm, sparse content |
| **Pitch** | Fundamental frequency (F0) mean and std | Pitch stability, vocal naturalness |
| **Stereo** | L/R RMS ratio, stereo width | Stereo imbalance, mono collapse |

The `source` label (AI or Human) comes from our manifest CSVs.



## 0. Install dependencies

Run this once per environment. `librosa` handles all the MIR analysis, `pyloudnorm` measures ITU-R BS.1770-5 loudness (the broadcast standard), `soundfile` reads the stereo channels separately.

In [ ]:
%pip install -q librosa soundfile pyloudnorm pandas numpy matplotlib seaborn tqdm

## 1. Imports and configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import pyloudnorm as pyln
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# -where everything lives 
BASE         = Path('dataset_collection_clean')
MANIFEST_DIR = BASE / 'manifests'
OUTPUT_DIR   = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_CSV   = OUTPUT_DIR / 'audio_features.csv'

# audio settings 
# 22050 Hz is commonly used for MIR feature extraction.
# It preserves frequencies up to approximately 11,025 Hz and reduces computation.
SAMPLE_RATE = 22050
N_MFCC      = 13      # number of MFCC coefficients — 13 is standard for music
N_CHROMA    = 12      # 12 chroma bins = 12 musical pitch classes (C, C#, D, ... B)
RESUME      = True    # if True, skip songs we already processed — safe to restart

# ---- manifest files mapped to their language and source label ----
# Each entry: CSV filename -> (language, source label, audio_path column name)
# We need the audio_path column because each CSV uses a slightly different column name
MANIFEST_MAP = {
    # human songs — these CSVs have an 'audio_path' column pointing to the mp3
    str(MANIFEST_DIR / 'english_human_final_with_audio.csv'): ('English', 'Human', 'audio_path'),
    str(MANIFEST_DIR / 'hindi_human_final_with_audio.csv')  : ('Hindi',   'Human', 'audio_path'),
    str(MANIFEST_DIR / 'chinese_human_final_with_audio.csv'): ('Chinese', 'Human', 'audio_path'),
    # AI songs — these CSVs use 'local_audio_path' instead
    str(BASE / 'ai' / 'english' / 'english_ai_manifest.csv'): ('English', 'AI', 'local_audio_path'),
    str(BASE / 'ai' / 'chinese' / 'chinese_ai_manifest.csv'): ('Chinese', 'AI', 'local_audio_path'),
    str(BASE / 'ai' / 'hindi'   / 'hindi_ai_300_suno_prompts.csv'): ('Hindi', 'AI', 'local_audio_path'),
}

print('Configuration ready.')
print('Output will be saved to:', OUTPUT_CSV)

## 2. Load manifests

All six manifest CSVs are loaded and stacked into one combined DataFrame. Each row represents one song. The `source` column (AI or Human) is the classification target.

Column names are normalised so that regardless of which CSV a row came from, every row has `song_id`, `title`, `artist`, `language`, `final_genre`, `source`, and `audio_path` available.

In [ ]:

# Load and Combine Manifests
frames = []

for csv_path,(language,source,audio_col) in MANIFEST_MAP.items():
    p = Path(csv_path)

    if not p.exists():
        print('NOT FOUND:',p)
        continue

    df = pd.read_csv(p,low_memory=False)

    if audio_col not in df.columns:
        print('MISSING AUDIO COLUMN:',p.name,audio_col)
        continue

    df['language'] = language
    df['source'] = source
    df['audio_path'] = df[audio_col].fillna('').astype(str).str.strip()

    if 'song_id' not in df.columns:
        if 'new_song_id' in df.columns:
            df['song_id'] = df['new_song_id']
        else:
            prefix = f'{language[:2].upper()}_{source.upper()}'
            df['song_id'] = [
                f'{prefix}_{i:04d}' for i in range(len(df))
            ]

    df['song_id'] = df['song_id'].fillna('').astype(str).str.strip()

    for col in ['title','artist','final_genre']:
        if col not in df.columns:
            df[col] = ''

    frames.append(df)
    print(f'Loaded {p.name}: {len(df)} rows [{language}/{source}]')

if not frames:
    raise FileNotFoundError('No manifest CSVs found. Check MANIFEST_MAP paths.')

manifest = pd.concat(frames,ignore_index=True)

print('\nCombined manifest:',len(manifest))
print(manifest.groupby(['language','source']).size().rename('count'))

print('\nMissing song IDs:',manifest['song_id'].eq('').sum())
print('Missing audio paths:',manifest['audio_path'].eq('').sum())
print('Duplicate song IDs:',manifest['song_id'].duplicated().sum())

## 3. Locate audio files

The `audio_path` column in the manifests should point directly to the MP3 file. Each path is verified to exist and any missing files are reported. Missing files can't be processed they'll be skipped in the extraction loop.

In [ ]:
# Resolve Audio File Paths
AUDIO_EXTS = {'.mp3','.wav','.flac','.m4a','.aac','.ogg'}

def resolve_audio_path(row):
    stored = str(row.get('audio_path','') or '').strip()

    if stored:
        for candidate in [Path(stored),BASE/stored]:
            if candidate.is_file():
                return str(candidate)

    sid = str(row.get('song_id','') or '').strip()
    source = str(row.get('source','')).casefold()
    language = str(row.get('language','')).casefold()

    if not sid:
        return None

    source_folder = 'human' if source=='human' else 'ai'
    search_dirs = [
        BASE/source_folder/language/'audio',
        BASE/source_folder/language
    ]

    for folder in search_dirs:
        if not folder.exists():
            continue

        for ext in AUDIO_EXTS:
            candidate = folder/f'{sid}{ext}'
            if candidate.is_file():
                return str(candidate)

        matches = sorted(
            file for file in folder.glob(f'{sid}*')
            if file.is_file() and file.suffix.casefold() in AUDIO_EXTS
        )

        if len(matches)==1:
            return str(matches[0])

        if len(matches)>1:
            print(f'AMBIGUOUS: {sid} → {len(matches)} files')
            return None

    return None

manifest['audio_path'] = manifest.apply(resolve_audio_path,axis=1)

found = manifest['audio_path'].notna().sum()
missing = manifest['audio_path'].isna().sum()

print('Audio found   :',found)
print('Audio missing :',missing)

if missing:
    print('\nSample missing files:')
    display(
        manifest.loc[
            manifest['audio_path'].isna(),
            ['song_id','title','artist','language','source']
        ].head(10)
    )

## 4. Feature extraction functions

Each function takes the loaded audio signal (`y`) and sample rate (`sr`) and returns a dictionary of features. Keeping them separate makes it straightforward to add or remove a group without touching the main loop.



### 4.1 Loudness (ITU-R BS.1770-5)

**LUFS** (Loudness Units relative to Full Scale) is the international broadcast loudness standard. Streaming platforms like Spotify normalise to around −14 LUFS. Tracks that sit higher than this have been aggressively compressed.

$$\text{Crest Factor} = \frac{\max(|x[n]|)}{\text{RMS}}$$

A crest factor below 6 dB means very little dynamic variation the loud and quiet parts are nearly the same level. This is a classic over-compression sign.

In [ ]:
def extract_loudness(y,sr):
    # ITU-R BS.1770 integrated loudness
    y = np.asarray(y,dtype=np.float32).flatten()
    meter = pyln.Meter(sr)

    try:
        lufs = float(meter.integrated_loudness(y))
    except Exception:
        lufs = np.nan

    peak = float(np.max(np.abs(y))) if len(y) else 0.0
    rms = float(np.sqrt(np.mean(y**2))) if len(y) else 0.0
    # Add a tiny value to avoid division-by-zero on silent tracks
    crest = peak/(rms+1e-9)  # tiny offset prevents division by zero on silence

    frame_rms = librosa.feature.rms(
        y=y,frame_length=2048,hop_length=512
    )[0]

    frame_db = librosa.amplitude_to_db(
        frame_rms,ref=1.0,top_db=None
    )

    clips = int(np.sum(np.abs(y)>=1.0))  # samples at or above digital full-scale
    silence_ratio = float(np.mean(frame_db<-60)) if len(frame_db) else np.nan  # fraction of frames below -60 dB
    dynamic_range = (
        float(np.percentile(frame_db,95)-np.percentile(frame_db,5))  # 90th-percentile loudness spread
        if len(frame_db) else np.nan
    )

    return {
        'lufs_integrated':round(lufs,3) if np.isfinite(lufs) else None,
        'sample_peak_dbfs':round(20*np.log10(peak+1e-9),3),
        'rms_db':round(20*np.log10(rms+1e-9),3),
        'crest_factor_db':round(20*np.log10(crest+1e-9),3),
        'dynamic_range_db':round(dynamic_range,3) if np.isfinite(dynamic_range) else None,
        'hard_clip_count':clips,
        'is_clipped':clips>0,
        'silence_ratio':round(silence_ratio,4) if np.isfinite(silence_ratio) else None
    }

### 4.2 Spectral features

These describe the frequency content of the audio at each moment in time.


In [ ]:
def extract_spectral(y,sr,n_mfcc=13,n_chroma=12):
    # Spectral flux measures how much the spectrum changes frame-to-frame.
    # We normalise each STFT frame by its total power before computing flux
    # so that loud and quiet sections contribute equally.
    stft = np.abs(librosa.stft(y))
    freqs = librosa.fft_frequencies(sr=sr)
    power = np.mean(stft**2,axis=1)
    total_p = np.sum(power)+1e-9

    centroid = librosa.feature.spectral_centroid(y=y,sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y,sr=sr)[0]
    flatness = librosa.feature.spectral_flatness(y=y)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y,sr=sr)[0]

    # Frame-normalised spectral flux: reduces amplitude/loudness influence
    frame_norm = np.linalg.norm(stft,axis=0,keepdims=True)  # per-frame magnitude for normalisation
    stft_norm = stft/(frame_norm+1e-9)  # level-normalise so flux is amplitude-independent
    flux = (
        np.mean(np.diff(stft_norm,axis=1)**2,axis=0)
        if stft_norm.shape[1]>1 else np.array([0.0])
    )

    contrast = librosa.feature.spectral_contrast(
        y=y,sr=sr,n_bands=6
    )

    mfccs = librosa.feature.mfcc(
        y=y,sr=sr,n_mfcc=n_mfcc
    )
    delta_mfccs = librosa.feature.delta(mfccs)

    chroma = librosa.feature.chroma_stft(
        y=y,sr=sr,n_chroma=n_chroma
    )

    out = {
        'spectral_centroid_mean':round(float(np.mean(centroid)),3),
        'spectral_centroid_std':round(float(np.std(centroid)),3),
        'spectral_rolloff_mean':round(float(np.mean(rolloff)),3),
        'spectral_flatness_mean':round(float(np.mean(flatness)),6),
        'spectral_bandwidth_mean':round(float(np.mean(bandwidth)),3),
        'spectral_flux_mean':round(float(np.mean(flux)),6),
        'spectral_contrast_mean':round(float(np.mean(contrast)),4),
        'band_energy_low':round(
            float(np.sum(power[freqs<250])/total_p),4  # sub-bass + bass band (<250 Hz)
        ),
        'band_energy_mid':round(
            float(np.sum(power[(freqs>=250)&(freqs<4000)])/total_p),4  # mid-range (250–4000 Hz)
        ),
        'band_energy_high':round(
            float(np.sum(power[freqs>=4000])/total_p),4  # high-frequency presence (>4000 Hz)
        ),
        'chroma_mean':round(float(np.mean(chroma)),4),
        'chroma_std':round(float(np.std(chroma)),4)
    }

    for i in range(n_mfcc):
        out[f'mfcc_{i+1}_mean'] = round(float(np.mean(mfccs[i])),4)
        out[f'mfcc_{i+1}_std'] = round(float(np.std(mfccs[i])),4)
        out[f'delta_mfcc_{i+1}_mean'] = round(
            float(np.mean(delta_mfccs[i])),4
        )
        out[f'delta_mfcc_{i+1}_std'] = round(
            float(np.std(delta_mfccs[i])),4
        )

    return out

### 4.3 Temporal features

These capture rhythm and timing characteristics.

**BPM** beats per minute, estimated from onset strength.
**Beat regularity** standard deviation of inter-beat intervals. Very low std indicates a more metronomically regular rhythm.
**Onset strength** how strong the note attacks are on average.
 **Zero crossing rate (ZCR)** how often the signal crosses zero. High ZCR = noisy or percussive content.

In [ ]:
def extract_temporal(y,sr):
    y = np.asarray(y,dtype=np.float32).flatten()

    if len(y)==0:
        return {
            'bpm':None,
            'beat_regularity_std':None,
            'beat_regularity_cv':None,
            'onset_strength_mean':None,
            'onset_strength_std':None,
            'zcr_mean':None,
            'zcr_std':None
        }

    # Onset strength is sensitive to overall loudness, so the signal is
    # amplitude-normalised first so that quiet and loud tracks are comparable.
    # Remove global amplitude differences
    y_norm = librosa.util.normalize(y)

    tempo,beats = librosa.beat.beat_track(
        y=y_norm,
        sr=sr
    )

    beat_times = librosa.frames_to_time(
        beats,
        sr=sr
    )

    if len(beat_times)>2:
        intervals = np.diff(beat_times)
        ibi_std = float(np.std(intervals))
        ibi_cv = ibi_std/(float(np.mean(intervals))+1e-9)  # coefficient of variation: std/mean beat interval
    else:
        ibi_std = None
        ibi_cv = None

    onset_env = librosa.onset.onset_strength(
        y=y_norm,
        sr=sr
    )

    zcr = librosa.feature.zero_crossing_rate(
        y_norm
    )[0]

    tempo_value = float(np.asarray(tempo).squeeze())

    return {
        'bpm':round(tempo_value,2),
        'beat_regularity_std':(
            round(ibi_std,4) if ibi_std is not None else None
        ),
        'beat_regularity_cv':(
            round(ibi_cv,4) if ibi_cv is not None else None
        ),
        'onset_strength_mean':round(float(np.mean(onset_env)),4),
        'onset_strength_std':round(float(np.std(onset_env)),4),
        'zcr_mean':round(float(np.mean(zcr)),6),
        'zcr_std':round(float(np.std(zcr)),6)
    }

### 4.4 Pitch features (F0)

Fundamental frequency (F0) is the perceived pitch of the vocal or dominant instrument. We extract it using YIN (librosa's `pyin` algorithm) and summarise the voiced frames.

High F0 std indicates expressive pitch variation
Very low F0 std indicates a more stable pitch contour

In [ ]:
def extract_pitch(y,sr):
    try:
        # C2 (65 Hz) to C7 (2093 Hz) covers the practical vocal and instrument range.
        # pyin also returns a voiced_flag per frame; only voiced frames are used
        # for statistics because unvoiced frames return NaN by design.
        f0,voiced_flag,_ = librosa.pyin(
            y,
            fmin=librosa.note_to_hz('C2'),
            fmax=librosa.note_to_hz('C7'),
            sr=sr
        )

        valid = voiced_flag & np.isfinite(f0)
        voiced_f0 = f0[valid]

        if len(voiced_f0)==0:
            raise ValueError('No voiced frames')

        # Estimate the track's tuning relative to equal temperament
        tuning = float(librosa.estimate_tuning(y=y,sr=sr))
        reference_a4 = 440.0*(2.0**(tuning/12.0))

        note_position = 12.0*np.log2(voiced_f0/reference_a4)  # distance in semitones from reference A4
        nearest_note = np.round(note_position)  # nearest equal-temperament note (integer semitones)
        nearest_frequency = reference_a4*(2.0**(nearest_note/12.0))

        deviations_cents = 1200.0*np.log2(voiced_f0/nearest_frequency)  # deviation in cents (100 cents = 1 semitone)
        abs_dev = np.abs(deviations_cents)

        return {
            'f0_mean':round(float(np.mean(voiced_f0)),3),
            'f0_std':round(float(np.std(voiced_f0)),3),
            'f0_voiced_ratio':round(float(np.mean(valid)),4),
            'tuning_offset_cents':round(tuning*100.0,2),
            'pitch_dev_mean_cents':round(float(np.mean(abs_dev)),3),
            'pitch_dev_std_cents':round(float(np.std(abs_dev)),3),
            'pitch_microtonal_ratio':round(float(np.mean(abs_dev>20)),4)  # fraction of voiced frames >20 cents off-pitch
        }

    except Exception:
        return {
            'f0_mean':None,
            'f0_std':None,
            'f0_voiced_ratio':None,
            'tuning_offset_cents':None,
            'pitch_dev_mean_cents':None,
            'pitch_dev_std_cents':None,
            'pitch_microtonal_ratio':None
        }

### 4.5 Stereo features

These require reading the original file as stereo (before any mono conversion). We measure the energy ratio between left and right channels and the stereo width. Differences in stereo treatment between AI and human recordings may appear in these measures.

In [ ]:
def extract_stereo(audio_path):
    try:
        # always_2d=True ensures even mono files return a 2-column array,
        # so the shape check on data.shape[1] is always safe.
        data,_=sf.read(str(audio_path),always_2d=True,dtype='float32')
        if data.shape[1]<2:
            return {'stereo_lr_ratio':None,'stereo_width':None,'is_mono':True}
        left=data[:,0]
        right=data[:,1]
        rms_l=float(np.sqrt(np.mean(left**2)))
        rms_r=float(np.sqrt(np.mean(right**2)))
        mid=(left+right)/2.0  # mono sum (correlated content)
        side=(left-right)/2.0  # side channel (stereo difference)
        mid_rms=float(np.sqrt(np.mean(mid**2)))
        side_rms=float(np.sqrt(np.mean(side**2)))
        return {
            'stereo_lr_ratio':round(rms_l/(rms_r+1e-9),4),
            'stereo_width':round(side_rms/(mid_rms+1e-9),4),  # >1 = wide stereo, ~0 = near-mono
            'is_mono':False
        }
    except Exception:
        return {'stereo_lr_ratio':None,'stereo_width':None,'is_mono':None}

print('All feature extraction functions defined.')

## 5. Spectrogram Preview AI vs Human

Before running the full extraction, a side-by-side comparison shows what the data looks like.
A **Mel spectrogram** shows how energy is distributed across frequencies over time it's basically
a heatmap of the song.


In [ ]:
# Mel Spectrogram Preview: AI vs Human
SPEC_AI=Path('dataset_collection_clean/ai/english/audio/AI_EN_FOLK_ACOUSTIC_001.mp3')
SPEC_HUMAN=Path('dataset_collection_clean/human/english/audio/HUMAN_EN_FOLK_ACOUSTIC_001.mp3')

fig,axes=plt.subplots(1,2,figsize=(16,5),sharey=True)
fig.suptitle('Mel Spectrogram Comparison: AI vs Human',fontsize=13,fontweight='bold',y=0.98)

image=None
for ax,fpath,label in zip(
    axes,
    [SPEC_AI,SPEC_HUMAN],
    ['AI Generated','Human Recording']
):
    if not fpath.exists():
        ax.text(0.5,0.5,f'File not found:\n{fpath.name}',ha='center',va='center',transform=ax.transAxes)
        ax.set_title(label,fontweight='bold')
        continue

    y_spec,sr_spec=librosa.load(str(fpath),sr=SAMPLE_RATE,mono=True,duration=30)
    mel=librosa.feature.melspectrogram(y=y_spec,sr=sr_spec,n_mels=128,fmax=sr_spec/2)
    mel_db=librosa.power_to_db(mel,ref=np.max,top_db=80)

    image=librosa.display.specshow(
        mel_db,sr=sr_spec,x_axis='time',y_axis='mel',
        fmax=sr_spec/2,cmap='magma',vmin=-80,vmax=0,ax=ax
    )

    ax.set_title(label,fontweight='bold')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Frequency (Mel scale)')

fig.subplots_adjust(left=0.07,right=0.91,bottom=0.15,top=0.84,wspace=0.09)

if image is not None:
    cbar_ax=fig.add_axes([0.925,0.15,0.012,0.69])
    fig.colorbar(
        image,cax=cbar_ax,
        format='%+2.0f dB',
        label='Relative power (dB)'
    )

spec_path=OUTPUT_DIR/'vis0_spectrogram_preview.png'
plt.savefig(spec_path,dpi=300,bbox_inches='tight')
plt.show()
print('Saved:',spec_path)

## 6. Main extraction loop

This part processes every audio file one by one.

**Time estimate:** ~90–120 seconds per song (dominated by pitch extraction via `pyin`).  
For 1800 songs that's roughly **45–60 hours on a CPU** let `RESUME=True` handle any interruptions.  
The tqdm progress bar below shows live ETA so you can see exactly how long is left.



In [ ]:
# Extract Only Missing Tracks
import time as _time

# RESUME=True means the loop skips songs already in the output CSV.
# Safe to interrupt and restart without re-processing completed songs.
RESUME=True

for fn in ['extract_loudness','extract_spectral','extract_temporal','extract_pitch','extract_stereo']:
    if fn not in globals():
        raise RuntimeError(f'{fn} not defined.')

existing=pd.DataFrame()
done=set()

if RESUME and OUTPUT_CSV.exists():
    existing=pd.read_csv(OUTPUT_CSV,low_memory=False)
    if 'ok' in existing.columns:
        successful=existing['ok'].astype(str).str.lower().eq('true')
        done=set(existing.loc[successful,'song_id'].astype(str))
    else:
        done=set(existing['song_id'].astype(str))
    existing=existing.drop_duplicates('song_id',keep='last').reset_index(drop=True)

to_do=manifest[manifest['audio_path'].notna()].copy()
to_do=to_do.drop_duplicates('song_id',keep='last')
to_do=to_do[~to_do['song_id'].astype(str).isin(done)].reset_index(drop=True)

print('Already processed:',len(done))
print('To process:',len(to_do))

records=[]
ok_count=0
err_count=0
start=_time.time()

for _,row in tqdm(to_do.iterrows(),total=len(to_do),desc='Extracting',unit='song'):
    apath=Path(str(row['audio_path']))
    sid=str(row.get('song_id','')).strip()

    rec={
        'song_id':sid,
        'title':row.get('title',''),
        'artist':row.get('artist',''),
        'language':str(row.get('language','')),
        'final_genre':str(row.get('final_genre','')),
        'source':str(row.get('source','')),
        'audio_path':str(apath),
        'ok':False,
        'error':''
    }

    try:
        if not apath.is_file():
            raise FileNotFoundError(apath)

        y,sr=librosa.load(str(apath),sr=SAMPLE_RATE,mono=True)

        rec.update(extract_loudness(y,sr))
        rec.update(extract_spectral(y,sr,N_MFCC,N_CHROMA))
        rec.update(extract_temporal(y,sr))
        rec.update(extract_pitch(y,sr))
        rec.update(extract_stereo(apath))

        rec['duration_seconds']=round(float(librosa.get_duration(y=y,sr=sr)),2)
        rec['ok']=True
        ok_count+=1

    except Exception as error:
        rec['error']=str(error)[:300]
        err_count+=1
        print(f'ERROR [{sid}]: {rec["error"]}')

    records.append(rec)

    new_rows=pd.DataFrame(records)
    out=pd.concat([existing,new_rows],ignore_index=True)
    out=out.drop_duplicates('song_id',keep='last').reset_index(drop=True)

    temp_path=OUTPUT_CSV.with_suffix('.tmp')
    out.to_csv(temp_path,index=False,encoding='utf-8-sig')
    temp_path.replace(OUTPUT_CSV)

elapsed=_time.time()-start

print('\nSuccessful:',ok_count)
print('Errors:',err_count)
print('Time: %.1f minutes'%(elapsed/60))
print('Saved:',OUTPUT_CSV)

In [ ]:
df=pd.read_csv(OUTPUT_CSV,low_memory=False)
valid_ids=set(manifest['song_id'].astype(str).str.strip())
df['song_id']=df['song_id'].astype(str).str.strip()
df=df[df['song_id'].isin(valid_ids)]
# keep='last' retains the most recent result if a song was processed
# more than once across resume runs.
df=df.drop_duplicates('song_id',keep='last').reset_index(drop=True)
df.to_csv(OUTPUT_CSV,index=False,encoding='utf-8-sig')

print('Rows:',len(df))
print('Unique IDs:',df['song_id'].nunique())
print('Missing:',len(valid_ids-set(df['song_id'])))
print('Extra:',len(set(df['song_id'])-valid_ids))
print(df.groupby(['language','source']).size())

In [ ]:
import numpy as np
import pandas as pd
import librosa
import scipy.stats as sp_stats
from pathlib import Path
from tqdm.auto import tqdm

CSV_PATH=Path('outputs/audio_features.csv')
if not CSV_PATH.exists():
    raise FileNotFoundError('audio_features.csv not found.')

df=pd.read_csv(CSV_PATH,low_memory=False)
ok=df['ok'].astype(str).str.lower().eq('true')
df_ok=df.loc[ok].copy()
print('Tracks to patch:',len(df_ok))

new_cols=['mel_kurtosis_mean','mel_kurtosis_std','mel_skewness_std']
for c in new_cols:
    df_ok[c]=np.nan

for i,row in tqdm(df_ok.iterrows(),total=len(df_ok),desc='Mel statistics patch'):
    try:
        y,sr=librosa.load(str(row['audio_path']),sr=SAMPLE_RATE,mono=True,duration=30)
        mel=librosa.feature.melspectrogram(y=y,sr=sr,n_mels=24,fmax=min(8000,sr/2))
        mel_db=librosa.power_to_db(mel+1e-12,ref=np.max)
        kurt=sp_stats.kurtosis(mel_db,axis=1,fisher=True,nan_policy='omit')
        skew=sp_stats.skew(mel_db,axis=1,nan_policy='omit')
        df_ok.at[i,'mel_kurtosis_mean']=round(float(np.nanmean(kurt)),3)
        df_ok.at[i,'mel_kurtosis_std']=round(float(np.nanstd(kurt)),3)
        df_ok.at[i,'mel_skewness_std']=round(float(np.nanstd(skew)),3)
    except Exception:
        continue

df.loc[df_ok.index,new_cols]=df_ok[new_cols]
df.drop(columns=['bark_kurtosis_mean','bark_kurtosis_std','clip_rate_per_sec'],errors='ignore',inplace=True)
df.to_csv(CSV_PATH,index=False,encoding='utf-8-sig')

print('Saved:',CSV_PATH)
print('Added:',new_cols)
print('Missing values:')
print(df[new_cols].isna().sum())

## 7. Summary table

A quick comparison of key features between AI and human songs. This is the first evidence table before any model training showing whether AI music is measurably different on raw audio features.

Larger mean differences indicate stronger separability for the classifier.

In [ ]:
df=pd.read_csv(OUTPUT_CSV,low_memory=False)
ok=df['ok'].astype(str).str.lower().eq('true')
df_ok=df.loc[ok].drop_duplicates('song_id',keep='last').copy()

print('Total successful unique songs:',len(df_ok))
print(df_ok.groupby(['language','source']).size().rename('count').reset_index().to_string(index=False))

KEY_FEATURES=[
    'lufs_integrated','crest_factor_db','dynamic_range_db',
    'spectral_centroid_mean','spectral_flatness_mean','spectral_flux_mean',
    'bpm','beat_regularity_std','silence_ratio','f0_std','stereo_width'
]
existing_kf=[f for f in KEY_FEATURES if f in df_ok.columns]

for col in existing_kf:
    df_ok[col]=pd.to_numeric(df_ok[col],errors='coerce')

summary=df_ok.groupby(['language','source'])[existing_kf].agg(['mean','std']).round(3)
summary.columns=['_'.join(c) for c in summary.columns]

print('\n=== Descriptive Feature Summary by Language and Source ===')
display(summary)

missing=df_ok[existing_kf].isna().mean().mul(100).round(2).rename('missing_percent')
print('\n=== Feature Missingness ===')
display(missing.to_frame())

## VIS 1 Feature distributions: LUFS, Crest Factor, BPM

Three histograms showing the most diagnostically useful features side by side. The reference lines show industry standard thresholds so we can see how AI and human tracks compare against real-world benchmarks.

In [ ]:
# Audio Feature Distributions by Language and Source
features=[
    ('lufs_integrated','Integrated loudness (LUFS)'),
    ('crest_factor_db','Crest factor (dB)'),
    ('bpm','Tempo (BPM)')
]
languages=['English','Hindi','Chinese']
palette={'AI':'#e05252','Human':'#4a90d9'}

fig,axes=plt.subplots(3,3,figsize=(15,11))

for r,language in enumerate(languages):
    lang_df=df_ok[df_ok['language']==language]
    for c,(col,xlabel) in enumerate(features):
        ax=axes[r,c]
        if col not in lang_df.columns:
            ax.set_title(f'{col} unavailable')
            ax.axis('off')
            continue
        values=pd.to_numeric(lang_df[col],errors='coerce')
        valid=values.dropna()
        if valid.empty:
            ax.set_title(f'{col} unavailable')
            ax.axis('off')
            continue
        bins=np.histogram_bin_edges(valid,bins=25)
        for source in ['AI','Human']:
            source_values=pd.to_numeric(
                lang_df.loc[lang_df['source']==source,col],
                errors='coerce'
            ).dropna()
            ax.hist(
                source_values,bins=bins,density=True,alpha=0.6,
                color=palette[source],edgecolor='white',label=source
            )
        ax.set_title(f'{language}: {xlabel}',fontsize=10,fontweight='bold')
        ax.set_xlabel(xlabel)
        ax.set_ylabel('Density')
        ax.legend(fontsize=8,frameon=False)
        ax.grid(axis='y',alpha=0.25)
        ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

fig.suptitle('Audio Feature Distributions by Language and Source',fontsize=14,fontweight='bold',y=0.995)
plt.tight_layout(rect=[0,0,1,0.98])
save_path=OUTPUT_DIR/'vis1_feature_distributions.png'
plt.savefig(save_path,bbox_inches='tight',dpi=300)
plt.show()
print('Saved:',save_path)

## Per-language feature boxplots (AI vs Human)

This checks whether the AI vs Human difference is consistent across all three languages. If the separation only shows up in one language, our features might be picking up on language or genre differences rather than genuine AI production characteristics.

In [ ]:
plot_features=[
    ('lufs_integrated','LUFS Integrated Loudness'),
    ('crest_factor_db','Crest Factor (dB)'),
    ('dynamic_range_db','Dynamic Range (dB)')
]
plot_features=[(f,lbl) for f,lbl in plot_features if f in df_ok.columns]

fig,axes=plt.subplots(1,len(plot_features),figsize=(6*len(plot_features),5))
if len(plot_features)==1:
    axes=[axes]

order=['English','Hindi','Chinese']

for ax,(feat,feat_label) in zip(axes,plot_features):
    temp=df_ok[['language','source',feat]].copy()
    temp[feat]=pd.to_numeric(temp[feat],errors='coerce')
    temp=temp.dropna(subset=[feat])

    sns.violinplot(
        data=temp,x='language',y=feat,hue='source',
        order=order,hue_order=['AI','Human'],
        palette=palette,split=True,inner='quartile',
        cut=0,linewidth=0.8,density_norm='width',ax=ax
    )

    ax.set_title(feat_label,fontsize=11,fontweight='bold')
    ax.set_xlabel('Language')
    ax.set_ylabel(feat_label)
    ax.legend(title='Source',fontsize=8,frameon=False)
    ax.grid(axis='y',alpha=0.25)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Per-Language Feature Distributions: AI vs Human',fontsize=13,fontweight='bold',y=1.02)
plt.tight_layout()
save_path=OUTPUT_DIR/'vis2_language_violinplots.png'
plt.savefig(save_path,bbox_inches='tight',dpi=300)
plt.show()
print('Saved:',save_path)

## Feature correlation heatmap

Before passing features into a model it helps to see which ones are redundant (correlation > 0.9). Highly correlated features don't add new information they just increase noise and computation. We limit to the first 20 features for readability.

In [ ]:
# Audio Feature Correlation Heatmap
SKIP={
    'song_id','title','artist','language','final_genre','source',
    'audio_path','ok','error','is_clipped','is_mono',
    'lufs_integrated','rms_db','sample_peak_dbfs',
    'hard_clip_count','clip_rate_per_sec',
    'pitch_dev_mean_cents','pitch_dev_std_cents',
    'pitch_microtonal_ratio','tuning_offset_cents'
}

preferred=[
    'crest_factor_db','dynamic_range_db','silence_ratio',
    'spectral_centroid_mean','spectral_centroid_std',
    'spectral_rolloff_mean','spectral_flatness_mean',
    'spectral_bandwidth_mean','spectral_flux_mean',
    'spectral_contrast_mean','band_energy_low',
    'band_energy_mid','band_energy_high',
    'bpm','beat_regularity_std','onset_strength_mean',
    'onset_strength_std','zcr_mean','zcr_std','stereo_width'
]

plot_cols=[]
for col in preferred:
    if col not in df_ok.columns or col in SKIP:
        continue
    values=pd.to_numeric(df_ok[col],errors='coerce')
    if values.isna().mean()<0.30 and values.nunique(dropna=True)>1:
        df_ok[col]=values
        plot_cols.append(col)

corr=df_ok[plot_cols].corr(method='spearman',min_periods=100)
mask=np.triu(np.ones_like(corr,dtype=bool))

fig,ax=plt.subplots(figsize=(14,11))
sns.heatmap(
    corr,mask=mask,annot=True,fmt='.2f',
    cmap='coolwarm',center=0,vmin=-1,vmax=1,
    linewidths=0.4,linecolor='white',
    annot_kws={'size':7},ax=ax
)
ax.set_title('Spearman Correlation Between Selected Audio Features',fontsize=12,fontweight='bold')
plt.xticks(rotation=45,ha='right',fontsize=8)
plt.yticks(rotation=0,fontsize=8)
plt.tight_layout()

save_path=OUTPUT_DIR/'vis3_correlation_heatmap.png'
plt.savefig(save_path,bbox_inches='tight',dpi=300)
plt.show()
print('Features shown:',len(plot_cols))
print('Saved:',save_path)

In [ ]:
# Clipping and loudness by source. Reported in Sections 2.2 and 4.1
# as the justification for excluding acquisition-linked features.
import pandas as pd
af = pd.read_csv(OUTPUT_DIR / 'audio_features.csv', low_memory=False)
for src in ['Human', 'AI']:
    g = af[af.source == src]
    print(f'{src:<6} clipped: {(g.is_clipped.astype(str).str.lower() == "true").mean():5.1%}   '
          f'mean LUFS: {pd.to_numeric(g.lufs_integrated, errors="coerce").mean():6.2f}')

h = pd.to_numeric(af.loc[af.source == "Human", "lufs_integrated"], errors="coerce").mean()
a = pd.to_numeric(af.loc[af.source == "AI", "lufs_integrated"], errors="coerce").mean()
print(f'\nLUFS difference: {abs(h - a):.2f}')

# Per-Track Quality Scoring with MuQ-Eval

## Why MuQ-Eval was used

The original proposal planned to create High, Medium and Low quality labels using
SongEval ratings. SongEval was not used because it does not provide genre labels,
while this study required balanced genres.

MuQ-Eval was therefore used as an alternative quality measure. It gives a separate
technical-quality score for each audio track and predicts expert Mean Opinion Scores.

Unlike Fréchet Audio Distance, which compares groups of audio files, MuQ-Eval can
score individual tracks. This makes it more suitable for identifying possible quality
differences within a single song.

The MuQ-Eval A1 model uses a frozen MuQ-310M audio encoder, attention pooling and a
two-layer neural-network prediction head.

## Output

This section creates:

`outputs/muq_eval_scores.csv`

The file contains one row for each track.

| Column | Meaning |
|---|---|
| `muq_mi_mean` | Average Musical Impression quality score from 1 to 5 |
| `muq_mi_std` | Variation in quality scores across the track |
| `muq_mi_slope` | Change in quality across the track; a negative value means the score decreases toward the end |
| `muq_mi_windows` | Quality scores for the individual audio windows |

The file is later used in `04_multimodal_fusion.ipynb` for technical-quality
analysis and the temporal-degradation analysis for RQ4.



### dependencies for MuQ-Eval


In [ ]:
%pip install datasets


In [ ]:
# `muq` is imported by the repo's src/encoders.py but is NOT in its requirements.txt
%pip install -q transformers peft omegaconf huggingface_hub muq



### fetch the MuQ-Eval repository


In [ ]:
# Download and Import MuQ-Eval
import io,sys,shutil,zipfile,urllib.request,importlib
from pathlib import Path

MUQ_REPO=Path('MuQ-Eval')
MODEL_FILE=MUQ_REPO/'src'/'model.py'
DATA_FILE=MUQ_REPO/'src'/'data.py'

if not MODEL_FILE.exists() or not DATA_FILE.exists():
    print('Downloading MuQ-Eval...')
    url='https://codeload.github.com/dgtql/MuQ-Eval/zip/refs/heads/main'
    temp_repo=Path('MuQ-Eval-main')

    if temp_repo.exists():
        shutil.rmtree(temp_repo)

    with urllib.request.urlopen(url,timeout=180) as response:
        data=response.read()

    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        archive.extractall('.')

    if not temp_repo.exists():
        raise FileNotFoundError('Downloaded MuQ-Eval folder was not found.')

    if MUQ_REPO.exists():
        shutil.rmtree(MUQ_REPO)

    temp_repo.rename(MUQ_REPO)
    print('Extracted:',MUQ_REPO)

repo_path=str(MUQ_REPO.resolve())
if repo_path in sys.path:
    sys.path.remove(repo_path)
sys.path.insert(0,repo_path)
importlib.invalidate_caches()

if not MODEL_FILE.exists() or not DATA_FILE.exists():
    raise FileNotFoundError('MuQ-Eval src/model.py or src/data.py is missing.')

from src.model import MusicQualityModel
from src.data import AudioProcessor

print('MuQ-Eval repo ready:',repo_path)

### load the A1 checkpoint


In [ ]:
import os,time,torch,librosa
from pathlib import Path
from omegaconf import OmegaConf

HF_CACHE_DIR=(Path.cwd()/'model_cache'/'huggingface').resolve()
HF_CACHE_DIR.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME']=str(HF_CACHE_DIR)
os.environ['HUGGINGFACE_HUB_CACHE']=str(HF_CACHE_DIR/'hub')
os.environ['TRANSFORMERS_CACHE']=str(HF_CACHE_DIR/'transformers')

from huggingface_hub import hf_hub_download

MUQ_HF_REPO='zhudi2825/MuQ-Eval-A1'
MUQ_SCORES_CSV=OUTPUT_DIR/'muq_eval_scores.csv'
MUQ_N_WINDOWS=5  # number of 10-second windows per track
MUQ_SKIP_EDGES_SEC=5.0  # seconds to skip at start and end (avoids silence/fade)
MUQ_MAX_THIS_RUN=1800
MUQ_CHECKPOINT=25  # save progress to CSV every 25 tracks
MUQ_THROTTLE=0.0
MUQ_DEVICE='cuda' if torch.cuda.is_available() else 'cpu'  # use GPU if available, fall back to CPU

print('Device:',MUQ_DEVICE)
print('Model cache:',HF_CACHE_DIR)

t0=time.time()
config_path=hf_hub_download(MUQ_HF_REPO,'config.yaml',cache_dir=str(HF_CACHE_DIR))
try:
    model_path=hf_hub_download(MUQ_HF_REPO,'best_model.pt',cache_dir=str(HF_CACHE_DIR))
except Exception:
    model_path=hf_hub_download(MUQ_HF_REPO,'model_state_dict.pt',cache_dir=str(HF_CACHE_DIR))

muq_cfg=OmegaConf.load(config_path)
base_path=MUQ_REPO/'configs'/'base.yaml'
if 'model' not in muq_cfg or 'encoder_id' not in muq_cfg.get('model',{}):
    muq_cfg=OmegaConf.merge(OmegaConf.load(base_path),muq_cfg)

muq_model=MusicQualityModel(muq_cfg)
checkpoint=torch.load(model_path,map_location='cpu',weights_only=False)

if isinstance(checkpoint,dict) and 'model_state' in checkpoint:
    state_dict=checkpoint['model_state']
elif isinstance(checkpoint,dict) and 'model_state_dict' in checkpoint:
    state_dict=checkpoint['model_state_dict']
else:
    state_dict=checkpoint

result=muq_model.load_state_dict(state_dict,strict=False)  # strict=False allows minor architecture mismatches
if result.missing_keys or result.unexpected_keys:
    raise RuntimeError(f'Checkpoint mismatch: missing={result.missing_keys[:5]}, unexpected={result.unexpected_keys[:5]}')

muq_model=muq_model.to(MUQ_DEVICE).eval()  # eval() disables dropout and batch-norm training mode
muq_processor=AudioProcessor(target_sr=int(muq_cfg.data.sample_rate),clip_samples=int(muq_cfg.data.clip_samples))
MUQ_CLIP_SEC=float(muq_cfg.data.clip_samples/muq_cfg.data.sample_rate)

print('Loaded in %.1f s | window %.1f s @ %d Hz'%(time.time()-t0,MUQ_CLIP_SEC,int(muq_cfg.data.sample_rate)))
print('Checkpoint:',Path(model_path).name)
print('MI will be used; TA will be ignored.')

Score one track at n evenly spaced windows; return aggregate stats + slope.


In [ ]:
def muq_score_track(audio_path,n_windows=MUQ_N_WINDOWS,skip_edges_sec=MUQ_SKIP_EDGES_SEC):
    y,sr=librosa.load(str(audio_path),sr=int(muq_cfg.data.sample_rate),mono=True)
    dur=len(y)/sr
    max_start=max(0.0,dur-MUQ_CLIP_SEC)
    # Skip the first and last few seconds to avoid intro silence
    # and outro fade-out, which would bias the quality score downward.
    usable_start=min(skip_edges_sec,max_start)
    usable_end=max(usable_start,max_start-skip_edges_sec)

    if usable_end<=usable_start:
        starts=[max_start/2.0]
    else:
        starts=np.linspace(usable_start,usable_end,n_windows).tolist()

    clips=[]
    clip_samples=int(MUQ_CLIP_SEC*sr)
    min_samples=clip_samples//2

    for start in starts:
        begin=int(start*sr)
        segment=y[begin:begin+clip_samples]
        if len(segment)>=min_samples:
            clips.append(muq_processor.process(segment,sr,mode='center'))

    if not clips:
        raise ValueError(f'No usable windows: duration={dur:.1f}s')

    batch=torch.stack(clips).to(MUQ_DEVICE)

    with torch.inference_mode():
        muq_model(batch)
        mi=muq_model._last_expected_scores['MI']

    mi=mi.detach().float().cpu().numpy().reshape(-1)

    if len(mi)>=3:  # need at least 3 windows to fit a meaningful slope
        positions=np.linspace(0.0,1.0,len(mi))
        slope=float(np.polyfit(positions,mi,1)[0])  # [0] = gradient of the fitted line
    else:
        slope=np.nan

    return {
        'duration_sec':round(float(dur),3),
        'n_windows':int(len(mi)),
        'muq_mi_mean':float(np.mean(mi)),
        'muq_mi_std':float(np.std(mi)),
        'muq_mi_min':float(np.min(mi)),
        'muq_mi_max':float(np.max(mi)),
        'muq_mi_first':float(mi[0]),
        'muq_mi_last':float(mi[-1]),
        'muq_mi_slope':slope,
        'muq_mi_windows':';'.join(f'{value:.4f}' for value in mi)
    }

print('Scoring function ready. Window=%.1f seconds.'%MUQ_CLIP_SEC)

In [ ]:
# Build Balanced MuQ-Eval Worklist
_muq_src=pd.read_csv(OUTPUT_CSV,low_memory=False)

if 'ok' in _muq_src.columns:
    ok=_muq_src['ok'].astype(str).str.lower().eq('true')
    _muq_src=_muq_src.loc[ok].copy()

_muq_src=_muq_src.drop_duplicates('song_id',keep='last')
_muq_src=_muq_src[_muq_src['audio_path'].notna()].copy()
_muq_src['audio_path']=_muq_src['audio_path'].astype(str).str.strip()
_muq_src=_muq_src[_muq_src['audio_path'].apply(lambda p:Path(p).is_file())]

keep=['song_id','source','language','final_genre','audio_path']
muq_work=_muq_src[[c for c in keep if c in _muq_src.columns]].copy()

parts=[]
for _,group in muq_work.groupby(['language','source'],sort=True):
    group=group.sample(frac=1,random_state=42).reset_index(drop=True)
    group['_round']=np.arange(len(group))
    parts.append(group)

muq_work=pd.concat(parts,ignore_index=True)
muq_work=muq_work.sort_values('_round',kind='stable').drop(columns='_round').reset_index(drop=True)

print('Worklist:',len(muq_work),'tracks')
print('Unique IDs:',muq_work['song_id'].nunique())
print(muq_work.groupby(['language','source']).size().rename('n').to_string())

In [ ]:
from tqdm.auto import tqdm
import time

if MUQ_SCORES_CSV.exists():
    previous=pd.read_csv(MUQ_SCORES_CSV,low_memory=False)
    previous=previous.drop_duplicates('song_id',keep='last').reset_index(drop=True)
    previous_ok=previous['ok'].astype(str).str.lower().eq('true')
    done_ids=set(previous.loc[previous_ok,'song_id'].astype(str))
    print('Successfully scored:',len(done_ids))
else:
    previous=pd.DataFrame()
    done_ids=set()

todo=muq_work[~muq_work['song_id'].astype(str).isin(done_ids)].head(MUQ_MAX_THIS_RUN).reset_index(drop=True)
print('Remaining overall:',len(muq_work)-len(done_ids))
print('This run:',len(todo))

new_records=[]
start=time.time()
ok_count=0
fail_count=0

def save_muq_checkpoint():
    new_df=pd.DataFrame(new_records)
    combined=new_df if previous.empty else pd.concat([previous,new_df],ignore_index=True)
    combined=combined.drop_duplicates('song_id',keep='last').reset_index(drop=True)
    temp_path=MUQ_SCORES_CSV.with_suffix('.tmp')
    combined.to_csv(temp_path,index=False,encoding='utf-8-sig')
    temp_path.replace(MUQ_SCORES_CSV)

for i,row in enumerate(tqdm(todo.to_dict('records'),desc='MuQ-Eval',unit='track'),1):
    record={
        'song_id':str(row['song_id']),
        'source':str(row.get('source','')),
        'language':str(row.get('language','')),
        'final_genre':str(row.get('final_genre','')),
        'ok':False,
        'error':''
    }
    try:
        record.update(muq_score_track(row['audio_path']))
        record['ok']=True
        ok_count+=1
    except Exception as error:
        record['error']=str(error)[:300]
        fail_count+=1
    new_records.append(record)
    if i%MUQ_CHECKPOINT==0 or i==len(todo):  # checkpoint after every 25 tracks and at the very end
        save_muq_checkpoint()
    if MUQ_THROTTLE>0:
        time.sleep(MUQ_THROTTLE)
    if i%50==0 and MUQ_DEVICE=='cuda':
        torch.cuda.empty_cache()  # release GPU memory that PyTorch caches between batches

if new_records:
    save_muq_checkpoint()

if MUQ_SCORES_CSV.exists():
    muq_scores_df=pd.read_csv(MUQ_SCORES_CSV,low_memory=False)
    muq_scores_df=muq_scores_df.drop_duplicates('song_id',keep='last')
    good=muq_scores_df['ok'].astype(str).str.lower().eq('true')
    muq_good=muq_scores_df.loc[good].copy()
    print('\nRun time: %.1f minutes'%((time.time()-start)/60))
    print('Successful this run:',ok_count)
    print('Failed this run:',fail_count)
    print('Total successful:',len(muq_good),'/',len(muq_work))
    print('Remaining:',len(muq_work)-len(muq_good))
    print(muq_good.groupby(['language','source']).size().rename('n').to_string())
    if fail_count:
        print('\nFailed tracks will be retried next time.')
else:
    print('No score file was created.')

In [ ]:
# MuQ-Eval Coverage Check
if 'muq_good' not in globals():
    raise RuntimeError('Run the MuQ-Eval scoring cell first.')

counts=muq_good.groupby(['language','source']).size().rename('scored')
expected=muq_work.groupby(['language','source']).size().rename('expected')
coverage=pd.concat([expected,counts],axis=1).fillna(0).astype(int)
coverage['remaining']=coverage['expected']-coverage['scored']
coverage['coverage_percent']=(coverage['scored']/coverage['expected']*100).round(1)

print('=== MuQ-Eval Coverage ===')
display(coverage.reset_index())

total_scored=len(muq_good)
total_expected=len(muq_work)
remaining=total_expected-total_scored

print('Total scored:',total_scored,'/',total_expected)
print('Remaining:',remaining)

if remaining==0:
    print('Status: Complete. MuQ-Eval results are ready for later analysis.')
else:
    print('Status: Partial. Rerun the scoring cell to process remaining tracks.')

print('\n=== Quality Score Summary ===')
summary=muq_good.groupby(['language','source'])['muq_mi_mean'].agg(['count','mean','std','median']).round(3)
display(summary)

print('Saved:',MUQ_SCORES_CSV)